# EDA-Streaming

### All Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import seaborn as sns

### Load Data

In [ ]:
df = pd.read_csv("spreadsheets/streaming.csv")
df.groupby("streaming").count()
df.info()

### EDA

#### Missing Values - Search

In [ ]:
streaming_na = df.isna().groupby(df["streaming"]).sum()

In [ ]:
plt.figure(figsize=(14, 7))
streaming_na.T.plot(
    kind='bar', 
    figsize=(14, 7), 
    title='Count of Missing Values by Variable, Compared by Streaming Service'
)

plt.ylabel('Number of Missing Values')
plt.xlabel('Dataset Variables')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Streaming Service')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#### Anomaly - Search

In [ ]:
df["country"].unique() # Clean

In [ ]:
df[(df["release_year"]<1928) | (df["release_year"]>2025)]["release_year"]

In [ ]:
df["rating"].unique() # Clean

In [ ]:
df["duration"].unique() # Clean and discretise

In [ ]:
df["listed_in"].unique() # Clean and discretise

In [ ]:
df["streaming"].unique()

#### Clean

##### Duration TV Shows - Seasons

In [ ]:
print("TV Show (Before):")
movie_durations_before = df[df["type"] == "TV Show"]["duration"]
print(f"Unique values (array): {movie_durations_before.unique()}")

count_before = movie_durations_before.nunique()
print(f"Number of unique values (Before): {count_before}")

df.loc[df["type"] == "TV Show", "duration"] = (
    df[df["type"] == "TV Show"]["duration"]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype(float)
    .astype('Int64')
)


print("\nAfter conversion:")
movie_durations_after = df[df["type"] == "TV Show"]["duration"]
print(f"Unique values (array): {movie_durations_after.unique()}")

count_after = movie_durations_after.nunique()
print(f"Number of unique values (After): {count_after}")

print("\nVerification:")
print(f"Is the count the same? {count_before == count_after}")

##### Duration Movies - Minutes

In [ ]:
print("Movies (Before):")
movie_durations_before = df[df["type"] == "Movie"]["duration"]
print(f"Unique values (array): {movie_durations_before.unique()}")

count_before = movie_durations_before.nunique()
print(f"Number of unique values (Before): {count_before}")

df.loc[df["type"] == "Movie", "duration"] = (
    df[df["type"] == "Movie"]["duration"]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype(float)
    .astype('Int64')
)


print("\nAfter conversion:")
movie_durations_after = df[df["type"] == "Movie"]["duration"]
print(f"Unique values (array): {movie_durations_after.unique()}")

count_after = movie_durations_after.nunique()
print(f"Number of unique values (After): {count_after}")

print("\nVerification:")
print(f"Is the count the same? {count_before == count_after}")

##### Rating

##### Listed in

##### Country

#### Discretise

##### Pre-discretise

In [ ]:
df['duration_class'] = pd.Series(np.nan, index=df.index, dtype='object')

##### Duration TV Shows - Seasons to Class

|   Class  | Num Seasons |
|----------|-------------|
|Miniseries|    N <= 1   |
|Short     | 2 <= N <= 3 |
|Medium    | 4 <= N <= 7 |
|Long      |    N >= 8   |

In [ ]:
bins = [0, 1, 3, 7, np.inf]
labels = ['Miniseries', 'Short', 'Medium', 'Long']

tv_show_mask = df['type'] == 'TV Show'

df.loc[tv_show_mask, 'duration_class'] = pd.cut(
    df.loc[tv_show_mask, 'duration'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

print(df[tv_show_mask]['duration_class'].value_counts(dropna=False))

##### Duration Movies - Minutes to Class

|   Class  | Time in Minutes |
|----------|-----------------|
|   Short  |     T <= 40     |
| Standard |  40 < T <= 100  |
|  Feature | 100 < T <= 180  |
|   Long   |     T > 180     |

In [ ]:
bins = [0, 40, 100, 180, np.inf]
labels = ['Short', 'Standard', 'Feature', 'Long']

movie_mask = df['type'] == 'Movie'
movie_durations = df.loc[movie_mask, 'duration'].dropna()
df.loc[movie_durations.index, 'duration_class'] = pd.cut(
    movie_durations,
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

print(df[movie_mask]['duration_class'].value_counts(dropna=False))

##### Listed in

##### Remove old cols

In [ ]:
df.drop('duration', axis=1, inplace=True)
df.info()

### Content Distribution Analysis

In [ ]:
type_counts = df['type'].value_counts()
type_percent = df['type'].value_counts(normalize=True) * 100

print("Types:")
print(type_counts)
print(f"\nPercent:")
print(type_percent)

plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='type')
plt.title('Movies vs TV Shows')
plt.show()

In [ ]:
country_counts = df['country'].value_counts().head(15)

plt.figure(figsize=(12, 8))
sns.barplot(x=country_counts.values, y=country_counts.index)
plt.title('Most common 15 countries by number of content')
plt.xlabel('Number of content')
plt.show()

print("MOst common 10 countries:")
print(country_counts.head(10))

In [ ]:
from collections import Counter

all_categories = []
for categories in df['listed_in'].dropna():
    cats = [cat.strip() for cat in categories.split(',')]
    all_categories.extend(cats)

category_counts = Counter(all_categories).most_common(20)

plt.figure(figsize=(12, 8))
categories, counts = zip(*category_counts)
sns.barplot(x=counts, y=categories)
plt.title('Most commmon 20 Categories/Genres')
plt.xlabel('Count of categories/Genres')
plt.tight_layout()
plt.show()

print("Most common 10 categories:")
for category, count in category_counts[:10]:
    print(f"{category}: {count}")

In [ ]:
print("Streaming distribution:")
print(df['streaming'].value_counts())

if len(df['streaming'].unique()) > 1:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='streaming', hue='type')
    plt.title('Content by streaming and type')
    plt.xticks(rotation=45)
    plt.show()